# *Nonlinear Arterial Hemodynamics*
## Chapter 7 companion — Nonlinear Arterial Hemodynamics

This notebook is the computational companion to Chapter 7. It does **not** introduce a fourth mechanism and does **not** combine constitutive anisotropy, geometry-parameterized dynamics, and wall compliance into one additive multiphysics equation.

Instead, it reproduces the chapter's branch-indexed synthesis,

$$
\mathcal L_{0,j}\mathbf q_j
+
\Delta\mathcal L_j\mathbf q_j
+
\mathcal N_j(\mathbf q_j,\mathbf q_j)
=
\mathbf f_j,
\qquad
j\in\{C,G,B\},
$$

and demonstrates computationally how the same quadratic algebra appears through three different branch-specific projections:

- local velocity--vorticity forcing;
- spatial spectral redistribution;
- zero-frequency mean transport.

The book nomenclature governs all reader-facing quantities.

**Execution:** a clean Google Colab runtime should reproduce all outputs using **Run all** with no manual choices.

# Chapter question

Chapter 7 asks what is genuinely common among the constitutive, geometry-parameterized, and compliant branches, and what must remain model-specific.

The notebook therefore has five responsibilities:

1. verify the common pairwise convolution algebra of a quadratic interaction;
2. reproduce one branch-specific nonlinear observable for each of $j=C,G,B$;
3. preserve each branch's own state space, operator, boundary conditions, and counterfactual;
4. compare non-zero output modes with the zero-frequency mean without treating them as one scalar diagnostic;
5. use VascuQuest only to supply physiological context for quantities that genuinely belong to the relevant branch.

No "unified arterial nonlinearity index" is constructed.

# Book mechanics used here

## One reference problem, three controlled departures

The book-level family is

$$
\mathcal L_{0,j}\mathbf q_j
+
\Delta\mathcal L_j\mathbf q_j
+
\mathcal N_j(\mathbf q_j,\mathbf q_j)
=
\mathbf f_j,
\qquad
j\in\{C,G,B\}.
$$

The index labels three separate equations:

- $C$: constitutive branch of Chapter 4;
- $G$: geometry-parameterized reduced branch of Chapter 5;
- $B$: compliant-boundary branch of Chapter 6.

The family is a comparison protocol. It is **not** an additive PDE of the form
$\Delta\mathcal L_C+\Delta\mathcal L_G+\Delta\mathcal L_B$.

## The nonlinear term as a harmonic mixer

For a real periodic branch state,

$$
\mathbf q_j(t)
=
\sum_{m=-M}^{M}
\widehat{\mathbf q}_{j,m}e^{im\Omega t},
\qquad
\widehat{\mathbf q}_{j,-m}
=
\overline{\widehat{\mathbf q}_{j,m}},
$$

and a bilinear form $\mathcal B_j$,

$$
\mathcal N_j(\mathbf q_j,\mathbf q_j)
=
\mathcal B_j(\mathbf q_j,\mathbf q_j).
$$

Therefore,

$$
\mathcal N_j(\mathbf q_j,\mathbf q_j)
=
\sum_{m=-2M}^{2M}
\left[
\sum_{m_1+m_2=m}
\mathcal B_j(
\widehat{\mathbf q}_{j,m_1},
\widehat{\mathbf q}_{j,m_2})
\right]
e^{im\Omega t}.
$$

Non-zero output indices represent harmonic transfer. The $m=0$ member is the zero-frequency projection.

## Quadratic interaction and projection hierarchy

The computational order is

$$
\text{state}
\rightarrow
\text{quadratic interaction}
\rightarrow
\begin{cases}
\text{local velocity--vorticity field},\\
\text{non-zero spectral modes},\\
\text{zero-frequency mean transport}.
\end{cases}
$$

The projection determines the observable.

# VascuQuest representation

VascuQuest/PWDB is used only for branch-relevant context.

For the selected virtual population, the notebook uses:

- age;
- heart rate;
- luminal-area waveform;
- pressure waveform where pressure--area phase is required;
- source geometry for one deterministic representative subject.

The common classical pulsatile scale is

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}},
\qquad
\Omega=\frac{2\pi}{T},
\qquad
\alpha=R\sqrt{\frac{\Omega}{\nu}}.
$$

The compliant branch additionally uses the directly observed moving-boundary ratio

$$
\epsilon_{\mathrm{VQ}}
=
\frac{\max_t|R(t)-R|}{R}.
$$

The geometry branch uses source segment length and inlet/outlet radius only as geometry descriptors. It does **not** infer the Chapter 5 map $\mathcal M_G$.

The constitutive branch uses the canonical Chapter 4 anisotropy coefficients as controlled model parameters. VascuQuest does **not** supply $\mathcal A_{ij}$.

These inputs are displayed together for context, but they are never combined into one physiological state vector or one fitted nonlinear model.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch07")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0
mu = 3.5e-3
nu = mu/rho

# Canonical branch parameters retained from Chapters 4--6.
A_ztheta = 0.1
A_thetaz = 0.1
A_thetatheta = 1.0

L_g = 4.0*3.141592653589793
alpha_G = 10.0
b_ref = 1.0
g_ref = 0.005
C_g = 0.1
b_G = b_ref*alpha_G**-2
g_G = g_ref*(1.0 + C_g/alpha_G)
kappa_c = 2.0

R_B = 4e-3
f_B = 1.2
Omega_B = 2.0*3.141592653589793*f_B
kR_B = 0.2
k_B = kR_B/R_B
P_B = 1.0

SITES = ["AorticRoot", "Carotid", "Femoral", "Radial"]

print("Working directory:", ROOT)

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.special import jv, iv
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire only the PWDB artifacts used by this synthesis notebook.
ARTIFACTS = [
    "model_configurations",
    "common_site_waveforms_csv",
    "geometry",
]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Deterministic subject metadata and representative subject.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
age_group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
age_group = age_group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_group.iloc[len(age_group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared plotting and PWDB waveform utilities.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "Carotid": "Carotid",
    "Femoral": "Femoral",
    "Radial": "Radial",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

def contiguous_prefix(*rows):
    finite = np.ones_like(np.asarray(rows[0], dtype=float), dtype=bool)
    for row in rows:
        finite &= np.isfinite(np.asarray(row, dtype=float))
    bad = np.flatnonzero(~finite)
    stop = int(bad[0]) if len(bad) else len(finite)
    if np.any(finite[stop:]):
        raise ValueError("Internal missing samples would alter waveform phase.")
    if stop < 16:
        raise ValueError("Insufficient contiguous waveform samples.")
    return tuple(np.asarray(row, dtype=float)[:stop] for row in rows)

print("Shared helpers ready.")

# The nonlinear term as a harmonic mixer

The chapter's common result can be tested without choosing any branch physics.

Take a real periodic scalar state containing the first two harmonics,

$$
q(t)
=
\Re\left\{
\widehat q_1e^{i\Omega t}
+
\widehat q_2e^{2i\Omega t}
\right\}.
$$

A quadratic observable $q^2$ must contain:

- a zero-frequency contribution;
- difference-frequency content;
- the original interaction frequencies;
- sum-frequency content up to $4\Omega$.

This calculation is algebraic. It is not a surrogate for any one branch operator $\mathcal B_j$.

In [ ]:
# Direct reconstruction-before-multiplication demonstration.
phase = np.linspace(0.0, 1.0, 2048, endpoint=False)

q1 = 1.0*np.exp(1j*0.35)
q2 = 0.45*np.exp(-1j*0.65)

q = np.real(
    q1*np.exp(2j*np.pi*phase)
    + q2*np.exp(4j*np.pi*phase)
)
q2_signal = q*q

coeff = np.fft.rfft(q2_signal)/len(q2_signal)
m = np.arange(len(coeff))
amp = 2.0*np.abs(coeff)
amp[0] = np.abs(coeff[0])

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(phase, q, color=BLACK, label=r"$q(t)$")
axes[0].plot(phase, q2_signal, color=DARK, linestyle="--", label=r"$q^2(t)$")
axes[0].set_xlabel(r"Normalized time, $t/T$")
axes[0].set_ylabel("Normalized value")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].stem(
    m[:6], amp[:6],
    linefmt="k-", markerfmt="ko", basefmt=" "
)
axes[1].set_xlabel(r"Output harmonic, $m$")
axes[1].set_ylabel("Quadratic-spectrum amplitude")
axes[1].set_xticks(m[:6])
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch07_quadratic_harmonic_mixer")
plt.show()

The $m=0$ coefficient and the non-zero output coefficients are generated by the same pairwise algebra.

What changes from branch to branch is the bilinear form, the state on which it acts, and the projection used to define the observable.

# Velocity--vorticity forcing is the local representation

For incompressible flow,

$$
(\mathbf u\cdot\nabla)\mathbf u
=
\nabla\left(\frac{|\mathbf u|^2}{2}\right)
-
\mathbf u\times\boldsymbol\omega,
$$

with

$$
\boldsymbol\ell
=
\mathbf u\times\boldsymbol\omega.
$$

The constitutive branch retains a linear harmonic solve but permits coupled axial and azimuthal velocity. The new pathway is

$$
\Delta\mathcal L_C
\longrightarrow
\{u_\theta,\omega_z\}
\longrightarrow
\boldsymbol\ell.
$$

The isotropic Womersley baseline already has $\omega_\theta$ and a radial Lamb-vector contribution. The relevant constitutive observable is therefore the increment relative to that baseline.

In [ ]:
# Compact Chapter 4 canonical harmonic solver for the Chapter 7 synthesis.
def solve_constitutive_harmonic(
    alpha=8.0,
    m=1,
    a_m=1.0,
    Azt=A_ztheta,
    Atz=A_thetaz,
    Att=A_thetatheta,
    epsilon=1e-5,
    tol=1e-7,
):
    lam = m*alpha**2
    C = np.array([[1.0, Azt], [Atz, Att]], dtype=float)
    Cinv = np.linalg.inv(C)
    x = np.linspace(epsilon, 1.0, 300)

    def ode(x, y):
        Uz = y[0] + 1j*y[4]
        Uzp = y[1] + 1j*y[5]
        Uth = y[2] + 1j*y[6]
        Uthp = y[3] + 1j*y[7]

        rhs_z = 1j*lam*Uz - a_m - Uzp/x
        rhs_th = (
            1j*lam*Uth
            - 2.0*Atz*Uzp/x
            - Att*Uthp/x
            + Att*Uth/x**2
        )
        second = Cinv @ np.vstack([rhs_z, rhs_th])

        out = np.empty_like(y)
        vals = [Uzp, second[0], Uthp, second[1]]
        for i, value in enumerate(vals):
            out[i] = value.real
            out[i+4] = value.imag
        return out

    def bc(ya, yb):
        residual = [
            ya[1] + 1j*ya[5],
            ya[2] + 1j*ya[6],
            yb[0] + 1j*yb[4],
            yb[2] + 1j*yb[6],
        ]
        return np.array(
            [z.real for z in residual]
            + [z.imag for z in residual]
        )

    y0 = np.zeros((8, len(x)))
    y0[0] = (1.0-x**2)/(1.0+alpha**2)
    y0[1] = -2.0*x/(1.0+alpha**2)

    sol = solve_bvp(
        ode, bc, x, y0,
        tol=tol, max_nodes=20000
    )
    if sol.status != 0:
        raise RuntimeError(sol.message)
    return sol

def eval_constitutive(sol, x):
    y = sol.sol(x)
    return (
        y[0]+1j*y[4],
        y[1]+1j*y[5],
        y[2]+1j*y[6],
        y[3]+1j*y[7],
    )

print("Constitutive branch solver ready.")

In [ ]:
# Constitutive branch: anisotropic increment and its nonlinear temporal spectrum.
xC = np.linspace(1e-5, 1.0, 700)

solC = solve_constitutive_harmonic()
Uz, Uzp, Uth, Uthp = eval_constitutive(solC, xC)

solC0 = solve_constitutive_harmonic(
    Azt=0.0, Atz=0.0, Att=1.0
)
Uz0, Uzp0, _, _ = eval_constitutive(solC0, xC)

phaseC = np.linspace(0.0, 1.0, 512, endpoint=False)
E1 = np.exp(2j*np.pi*phaseC)[:, None]

uz = np.real(E1*Uz[None, :])
uzp = np.real(E1*Uzp[None, :])
uth = np.real(E1*Uth[None, :])
uthp = np.real(E1*Uthp[None, :])

omega_z = uthp + uth/xC[None, :]
ell_an = uth*omega_z + uz*uzp

uz0 = np.real(E1*Uz0[None, :])
uzp0 = np.real(E1*Uzp0[None, :])
ell_iso = uz0*uzp0

delta_ell = ell_an - ell_iso

# Select a near-wall location without evaluating at the boundary.
jC = int(np.argmin(np.abs(xC-0.90)))
signalC = delta_ell[:, jC]
specC = np.fft.rfft(signalC)/len(signalC)
mC = np.arange(len(specC))
ampC = 2.0*np.abs(specC)
ampC[0] = np.abs(specC[0])

fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.1))

axes[0].plot(
    xC, delta_ell[0],
    color=BLACK, label=r"$t/T=0$"
)
axes[0].plot(
    xC, delta_ell[len(phaseC)//4],
    color=DARK, linestyle="--", label=r"$t/T=0.25$"
)
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"Dimensionless $\Delta\ell_r$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].stem(
    mC[:6], ampC[:6],
    linefmt="k-", markerfmt="ko", basefmt=" "
)
axes[1].set_xlabel(r"Output harmonic, $m$")
axes[1].set_ylabel(r"Spectrum of $\Delta\ell_r$ at $x=0.9$")
axes[1].set_xticks(mC[:6])
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch07_constitutive_local_projection")
plt.show()

This is the Chapter 7 **local projection**.

The branch-specific state contains coupled velocity components. The nonlinear observable is then constructed from the real velocity and vorticity fields. Its temporal spectrum contains information that is not present in the individual linear harmonic amplitude alone.

The constitutive result is not combined numerically with the geometry or compliance branches.

# Harmonic interaction becomes spectral redistribution

The geometry branch uses the reduced axial state $\widetilde a(\zeta,s)$.

Its nonlinear term is

$$
\widetilde a\,\widetilde a_\zeta,
$$

whose Fourier transform is

$$
\mathcal F\{
\widetilde a\,\widetilde a_\zeta
\}_\kappa
=
\frac{i\kappa}{2}
\sum_{\kappa_1+\kappa_2=\kappa}
\widehat a_{\kappa_1}\widehat a_{\kappa_2}.
$$

The implemented periodic model satisfies

$$
\frac{dI_2}{ds}
=
-2g_{\mathrm{avg}}
\int_0^{L_g}
\widetilde a
(-\partial_\zeta^2)^{1/2}
\widetilde a\,d\zeta
\le0.
$$

Thus nonlinear transfer can broaden the spatial spectrum while total quadratic energy decreases.

In [ ]:
# Compact Chapter 5 split-Fourier solver.
def geometry_grid(N):
    zeta = np.arange(N)*L_g/N
    kappa = 2.0*np.pi*np.fft.fftfreq(N, d=L_g/N)
    mode_number = np.fft.fftfreq(N)*N
    return zeta, kappa, mode_number

def geometry_initial(zeta):
    return (
        np.sin(0.5*zeta)
        + 0.3*np.sin(zeta)
        + 0.1*np.sin(1.5*zeta)
    )

def geometry_diag(a_hat, kappa):
    N = len(a_hat)
    coeff = a_hat/N
    I2 = L_g*np.sum(np.abs(coeff)**2)

    low = (
        (np.abs(kappa) > 1e-14)
        & (np.abs(kappa) <= kappa_c+1e-14)
    )
    high = np.abs(kappa) > kappa_c+1e-14

    Elow = L_g*np.sum(np.abs(coeff[low])**2)
    Ehigh = L_g*np.sum(np.abs(coeff[high])**2)
    return I2, Elow, Ehigh, Ehigh/Elow

def run_geometry_branch(
    N=256,
    dt=2e-3,
    s_end=10.0,
    nonlinear=True,
):
    zeta, kappa, mode_number = geometry_grid(N)
    a_hat = np.fft.fft(geometry_initial(zeta))

    linear = 1j*b_G*kappa**3 - g_G*np.abs(kappa)
    Ehalf = np.exp(linear*dt/2.0)
    mask = np.abs(mode_number) <= N/3.0
    a_hat[~mask] = 0.0

    def rhs(h):
        if not nonlinear:
            return np.zeros_like(h)
        a = np.fft.ifft(h).real
        out = -0.5j*kappa*np.fft.fft(a*a)
        out[~mask] = 0.0
        return out

    nsteps = int(round(s_end/dt))
    history = []

    for step in range(nsteps+1):
        if step % 50 == 0 or step == nsteps:
            I2, Elow, Ehigh, Rspec = geometry_diag(a_hat, kappa)
            history.append((step*dt, I2, Rspec))

        if step == nsteps:
            break

        a_hat = Ehalf*a_hat
        a_hat[~mask] = 0.0

        k1 = rhs(a_hat)
        k2 = rhs(a_hat+0.5*dt*k1)
        k3 = rhs(a_hat+0.5*dt*k2)
        k4 = rhs(a_hat+dt*k3)
        a_hat += dt*(k1+2*k2+2*k3+k4)/6.0
        a_hat[~mask] = 0.0

        a_hat = Ehalf*a_hat

    return zeta, kappa, a_hat, np.asarray(history)

print("Geometry branch solver ready.")

In [ ]:
# Geometry branch: redistribution with and without quadratic mode coupling.
zG, kG, hatG, histG = run_geometry_branch(nonlinear=True)
_, _, hatG_linear, histG_linear = run_geometry_branch(nonlinear=False)

coeffG = hatG/len(hatG)
coeffG_linear = hatG_linear/len(hatG_linear)
positive = (kG > 0) & (kG <= 8.0)

fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.1))

axes[0].stem(
    kG[positive],
    L_g*np.abs(coeffG[positive])**2,
    linefmt="k-", markerfmt="ko", basefmt=" ",
    label="quadratic coupling retained"
)
axes[0].stem(
    kG[positive],
    L_g*np.abs(coeffG_linear[positive])**2,
    linefmt="0.65", markerfmt="o", basefmt=" ",
    label="quadratic coupling removed"
)
axes[0].axvline(kappa_c, color=LIGHT, linestyle=":", linewidth=1.0)
axes[0].set_xlabel(r"Reduced axial wavenumber, $\kappa$")
axes[0].set_ylabel(r"Modal contribution to $I_2$ at $s=10$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(
    histG[:,0], histG[:,1],
    color=BLACK, label=r"$I_2$"
)
axR = axes[1].twinx()
axR.plot(
    histG[:,0], histG[:,2],
    color=DARK, linestyle="--",
    label=r"$\mathcal R_{\mathrm{spec}}$"
)
axes[1].set_xlabel(r"Dimensionless time, $s$")
axes[1].set_ylabel(r"$I_2$")
axR.set_ylabel(r"$\mathcal R_{\mathrm{spec}}$")
axes[1].spines["top"].set_visible(False)
axR.spines["top"].set_visible(False)

lines = axes[1].get_lines() + axR.get_lines()
axes[1].legend(
    lines, [line.get_label() for line in lines],
    frameon=False, loc="center right"
)

fig.tight_layout(w_pad=1.35)
save_figure(fig, "ch07_geometry_spectral_projection")
plt.show()

This is the Chapter 7 **spatial spectral projection**.

Removing the quadratic term prevents convolution-driven population of new spatial modes. With the nonlinearity retained, the spectrum broadens even while the total quadratic energy follows the non-increasing balance imposed by positive damping.

The relevant output variable is $\kappa$, not the temporal harmonic number $m$.

# The zero-frequency projection is steady streaming

For the compliant branch,

$$
\mathbf u_1
=
\Re\left\{
\widehat{\mathbf u}_1e^{-i\Omega t}
\right\},
$$

and

$$
\left\langle
(\mathbf u_1\cdot\nabla)\mathbf u_1
\right\rangle
=
\frac12
\Re\left\{
\overline{\widehat{\mathbf u}_1}
\cdot
\nabla\widehat{\mathbf u}_1
\right\}.
$$

This is the zero-frequency member of the same pairwise quadratic algebra.

The resulting mean equation is

$$
\mu\nabla^2\langle\mathbf u_2\rangle
-
\nabla\langle p_2\rangle
=
\rho
\left\langle
(\mathbf u_1\cdot\nabla)\mathbf u_1
\right\rangle,
$$

together with the Taylor-expanded moving-wall boundary condition.

In [ ]:
# Compact Chapter 6 first-order compliant mode and second-order mean solve.
def compliant_first_order(r, R, Omega, k, P):
    r = np.asarray(r, dtype=float)
    lambda_C = np.sqrt(1j*rho*Omega/mu - k**2)
    A = k*P/(Omega*rho)

    Cpsi = (
        -A*iv(0, k*R)
        /(lambda_C*jv(0, lambda_C*R))
    )

    uz = A*iv(0, k*r) + Cpsi*lambda_C*jv(0, lambda_C*r)
    ur = -1j*(
        A*iv(1, k*r)
        + k*Cpsi*jv(1, lambda_C*r)
    )
    uz_r = (
        A*k*iv(1, k*r)
        - Cpsi*lambda_C**2*jv(1, lambda_C*r)
    )

    eta_hat = (
        A*iv(1, k*R)
        + k*Cpsi*jv(1, lambda_C*R)
    )/Omega

    return uz, ur, uz_r, eta_hat

def run_compliance_branch(N=1600, remove_sources=False):
    r = np.linspace(0.0, R_B, N)
    dr = r[1]-r[0]

    uz, ur, uz_r, eta_hat = compliant_first_order(
        r, R_B, Omega_B, k_B, P_B
    )

    forcing = 0.5*np.real(
        np.conj(ur)*uz_r
        + np.conj(uz)*(1j*k_B*uz)
    )

    if remove_sources:
        forcing[:] = 0.0
        wall_value = 0.0
    else:
        wall_value = -0.5*np.real(
            np.conj(eta_hat)*uz_r[-1]
        )

    rhs = rho*forcing/mu

    A = lil_matrix((N, N), dtype=float)
    A[0,0] = -4.0/dr**2
    A[0,1] = 4.0/dr**2

    for i in range(1, N-1):
        ri = r[i]
        A[i,i-1] = 1.0/dr**2 - 1.0/(2.0*ri*dr)
        A[i,i] = -2.0/dr**2
        A[i,i+1] = 1.0/dr**2 + 1.0/(2.0*ri*dr)

    A[-1,-1] = 1.0
    rhs[-1] = wall_value

    u2 = spsolve(A.tocsr(), rhs)
    Qstream = 2.0*np.pi*np.trapezoid(r*u2, r)

    return r, forcing, u2, Qstream, wall_value

print("Compliance branch solver ready.")

In [ ]:
# Compliance branch: zero-frequency forcing and mean response.
rB, forcingB, u2B, QB, wallB = run_compliance_branch(
    N=1800, remove_sources=False
)
_, forcingB0, u2B0, QB0, wallB0 = run_compliance_branch(
    N=1800, remove_sources=True
)

fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.1))

axes[0].plot(
    rB/R_B, rho*forcingB,
    color=BLACK
)
axes[0].set_xlabel(r"Normalized radius, $r/R$")
axes[0].set_ylabel(
    r"Zero-frequency forcing, "
    r"$\rho\langle(\mathbf u_1\cdot\nabla)u_{1z}\rangle$"
)
clean_axes(axes[0])

axes[1].plot(
    rB/R_B, u2B,
    color=BLACK, label="quadratic sources retained"
)
axes[1].plot(
    rB/R_B, u2B0,
    color=DARK, linestyle="--",
    label="quadratic sources removed"
)
axes[1].set_xlabel(r"Normalized radius, $r/R$")
axes[1].set_ylabel(r"$\langle u_{2z}\rangle$ (m s$^{-1}$)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch07_compliance_zero_frequency_projection")
plt.show()

print("Case C streaming flux:", QB, "m^3/s")
print("Both second-order sources removed:", QB0, "m^3/s")

This is the Chapter 7 **zero-frequency projection**.

The first-order oscillatory state has zero mean, but its quadratic self-interaction produces a non-zero period-averaged forcing. Together with the second-order moving-wall boundary condition, it generates a persistent mean field.

When both second-order quadratic sources are removed, the mean field collapses to zero.

# The same nonlinear algebra produces different observables

The three calculations above should not be collapsed into a common numerical scale.

They occupy different mathematical locations:

| Branch | State | Quadratic projection | Observable |
|---|---|---|---|
| $C$ | coupled radial harmonic velocity | local velocity--vorticity product | $\Delta\ell_r$, $\rho\Delta\ell_r$ |
| $G$ | reduced axial amplitude | spatial Fourier convolution | $|\widehat a_\kappa|^2$, $\mathcal R_{\mathrm{spec}}$, $I_2$ |
| $B$ | finite-$k$ compliant first-order field | zero-frequency period average | $\langle\mathbf u_2\rangle$, $\langle Q_{\mathrm{stream}}\rangle$ |

The commonality is the pairwise quadratic algebra. The units, state spaces, boundary conditions, and observables remain branch-specific.

In [ ]:
# Compact numerical counterfactual audit across the three branches.
#
# These are deliberately branch-normalized yes/no-style checks, not a
# cross-branch physical ranking.
constitutive_ratio = np.max(np.abs(Uth))/np.max(np.abs(Uz))
constitutive_ratio_off = 0.0

Rspec_full = histG[-1,2]
Rspec_linear = histG_linear[-1,2]

Q_full = QB
Q_off = QB0

audit = pd.DataFrame({
    "branch": ["C", "G", "B"],
    "mechanism_retained": [
        constitutive_ratio,
        Rspec_full,
        Q_full,
    ],
    "mechanism_removed": [
        constitutive_ratio_off,
        Rspec_linear,
        Q_off,
    ],
    "quantity": [
        "max|U_theta|/max|U_z|",
        "R_spec(10)",
        "Q_stream",
    ],
})
audit.to_csv(
    DATA_DIR / "ch07_branch_counterfactual_audit.csv",
    index=False
)
display(audit)

The counterfactual table is intentionally not plotted on one axis because the three quantities have different units and meanings.

Its role is simply to verify the Chapter 7 comparison protocol:

- scalar isotropic viscosity removes the constitutive transverse channel;
- removing quadratic mode coupling suppresses geometry-branch spectral transfer;
- removing both second-order quadratic sources removes compliant-branch mean transport.

# VascuQuest exploration

Chapter 7 does not license a combined patient-specific nonlinear solver. VascuQuest is therefore used to show **branch-relevant context without branch fusion**.

Three quantities are displayed:

1. $\alpha$, the classical pulsatile scale shared by the reference problem;
2. representative source geometry descriptors relevant to the motivation for the Chapter 5 map $\mathcal M_G$;
3. $\epsilon_{\mathrm{VQ}}$, the observed wall-motion ratio relevant to the Chapter 6 perturbation ordering.

The constitutive coefficients $\mathcal A_{ij}$ remain controlled book parameters and are not inferred from VascuQuest.

In [ ]:
# Build population alpha and moving-wall context.
meta = subject_meta.set_index("subject_id")
context_rows = []

for site in SITES:
    ids_a, A_matrix = load_waveform_matrix(site, "A")

    for sid, A_row in zip(ids_a, A_matrix):
        if sid not in meta.index:
            continue
        try:
            (A_values,) = contiguous_prefix(A_row)
        except ValueError:
            continue

        R_t = np.sqrt(A_values/np.pi)
        R_mean = float(np.mean(R_t))
        epsilon_vq = float(
            np.max(np.abs(R_t-R_mean))/R_mean
        )

        heart_rate = float(meta.loc[sid, "heart_rate_bpm"])
        Omega = 2.0*np.pi*heart_rate/60.0
        alpha = R_mean*np.sqrt(Omega/nu)

        context_rows.append({
            "subject_id": sid,
            "age_years": float(meta.loc[sid, "age_years"]),
            "site": site,
            "alpha": alpha,
            "epsilon_VQ": epsilon_vq,
        })

context_df = pd.DataFrame(context_rows)
context_df.to_csv(
    DATA_DIR / "ch07_vascuquest_population_context.csv",
    index=False
)

# Geometry descriptors for the same deterministic representative subject.
geo_result = session.geometry(subject=representative_subject)
geo_rows = []

for seg in geo_result.values:
    Rm = 0.5*(seg.inlet_radius_m + seg.outlet_radius_m)
    geo_rows.append({
        "segment_id": seg.segment_id,
        "R_m": Rm,
        "L_m": seg.length_m,
        "L_over_Rm": seg.length_m/Rm,
        "taper_ratio":
            (seg.inlet_radius_m-seg.outlet_radius_m)/Rm,
    })

geo_df = pd.DataFrame(geo_rows)
geo_df.to_csv(
    DATA_DIR / "ch07_representative_geometry_context.csv",
    index=False
)

fig, axes = plt.subplots(1, 3, figsize=(10.1, 3.05))

# (a) alpha by site
alpha_groups = [
    context_df.loc[context_df["site"] == site, "alpha"].to_numpy()
    for site in SITES
]
axes[0].boxplot(
    alpha_groups,
    labels=[SITE_LABELS[s] for s in SITES],
    showfliers=False,
    whis=(5,95),
    widths=0.55,
    medianprops={"color": BLACK},
    boxprops={"color": BLACK},
    whiskerprops={"color": DARK},
    capprops={"color": DARK},
)
axes[0].tick_params(axis="x", rotation=50)
axes[0].set_ylabel(r"Womersley number, $\alpha$")
clean_axes(axes[0])

# (b) representative geometry
axes[1].scatter(
    geo_df["L_over_Rm"],
    geo_df["taper_ratio"],
    s=18,
    facecolors="none",
    edgecolors=BLACK,
    linewidths=0.7,
)
axes[1].set_xscale("log")
axes[1].axhline(0.0, color=LIGHT, linewidth=0.8)
axes[1].set_xlabel(r"Representative geometry, $L/R_m$")
axes[1].set_ylabel(r"$(R_{\mathrm{in}}-R_{\mathrm{out}})/R_m$")
clean_axes(axes[1])

# (c) wall-motion ratio
eps_groups = [
    context_df.loc[
        context_df["site"] == site, "epsilon_VQ"
    ].to_numpy()
    for site in SITES
]
axes[2].boxplot(
    eps_groups,
    labels=[SITE_LABELS[s] for s in SITES],
    showfliers=False,
    whis=(5,95),
    widths=0.55,
    medianprops={"color": BLACK},
    boxprops={"color": BLACK},
    whiskerprops={"color": DARK},
    capprops={"color": DARK},
)
axes[2].tick_params(axis="x", rotation=50)
axes[2].set_ylabel(r"Wall-motion ratio, $\epsilon_{\mathrm{VQ}}$")
clean_axes(axes[2])

fig.tight_layout(w_pad=1.2)
save_figure(fig, "ch07_vascuquest_branch_context")
plt.show()

The three panels are deliberately **not** combined into one parameter or score.

- $\alpha$ belongs to the classical pulsatile scale and can enter branch parameterizations already defined in the book.
- The geometry descriptors document resolved arterial shape information but do not define $\mathcal M_G$.
- $\epsilon_{\mathrm{VQ}}$ measures observed wall motion and informs the perturbation ordering, but does not identify the Kelvin--Voigt wall coefficients.
- No VascuQuest quantity in this notebook identifies $\mathcal A_{ij}$.

This is the maximum branch comparison supported without constructing a new multiphysics model.

In [ ]:
# Descriptive age context, kept branch-separated.
age_context = (
    context_df
    .groupby(["age_years", "site"])
    .agg(
        alpha_median=("alpha", "median"),
        epsilon_median=("epsilon_VQ", "median"),
    )
    .reset_index()
)

styles = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

for site, (gray, ls, marker) in zip(SITES, styles):
    sub = age_context.loc[age_context["site"] == site]
    axes[0].plot(
        sub["age_years"], sub["alpha_median"],
        color=gray, linestyle=ls, marker=marker,
        markersize=4, label=SITE_LABELS[site]
    )
    axes[1].plot(
        sub["age_years"], sub["epsilon_median"],
        color=gray, linestyle=ls, marker=marker,
        markersize=4, label=SITE_LABELS[site]
    )

axes[0].set_xlabel("PWDB source age (years)")
axes[0].set_ylabel(r"Median $\alpha$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel("PWDB source age (years)")
axes[1].set_ylabel(r"Median $\epsilon_{\mathrm{VQ}}$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.2)
save_figure(fig, "ch07_age_group_branch_context")
plt.show()

The age comparison remains descriptive. Age is a grouping variable in PWDB; it is not a constitutive law.

No causal statement is made about age-dependent anisotropy, geometry-induced spectral transfer, or compliant streaming.

# What is genuinely unified, and what remains model-specific

The notebook supports four common statements.

**First:** every extension is measured against the classical Womersley reference or its appropriate reduced descendant.

**Second:** the added mechanism is assigned to a term, operator, or boundary condition before its consequence is interpreted.

**Third:** quadratic products generate information absent from the separate linear amplitudes. The same algebraic fact appears as:

- temporal cross-harmonic content in the Lamb-vector observable;
- spatial convolution and redistribution among $\kappa$ modes;
- the $m=0$ period-averaged forcing that drives steady streaming.

**Fourth:** observables remain representation-specific.

The following remain model-specific:

- the constant reduced anisotropy tensor of the constitutive branch;
- the effective-medium reduced equation and geometry-to-coefficient interface of the geometry branch;
- the thin Kelvin--Voigt wall and finite-$k$ compliant mode of the boundary branch.

The notebook therefore demonstrates a common nonlinear architecture without constructing a common closed multiphysics model.

# What the reader should learn

1. **Chapter 7 is a synthesis chapter, not a new governing-model chapter.**

2. **The common structure is pairwise quadratic interaction.** In a harmonic representation, every output index is assembled from pairs whose indices sum to that output.

3. **Zero frequency is not algebraically separate from harmonic transfer.** It is the $m=0$ member of the same convolution structure.

4. **Local, temporal, spatial, and mean projections answer different questions.** A Lamb-vector force density, a temporal nonlinear spectrum, a spatial redistribution ratio, and a streaming flux are not interchangeable.

5. **The constitutive branch modifies the admissible velocity/vorticity state before the nonlinear product is formed.**

6. **The geometry branch makes spatial mode coupling explicit.** Broadening can occur while the total quadratic energy decreases.

7. **The compliant branch turns oscillatory self-interaction into a zero-frequency forcing and second-order mean response.**

8. **Counterfactual recovery is part of the synthesis.** Removing each branch mechanism must recover its reference limit.

9. **VascuQuest can supply branch-relevant physiological context without licensing branch fusion.**

10. **No scalar 'nonlinearity score' follows from Chapter 7.** Such a quantity would collapse observables with different dimensions, state spaces, and mechanical meanings.

# Chapter-enrichment candidates

The notebook produces six principal figures.

**Candidate 1 — quadratic harmonic mixer.**  
Strong conceptual candidate if Chapter 7 needs a direct computational visualization of Eq. (quadratic convolution), especially the simultaneous appearance of non-zero and zero-frequency outputs.

**Candidate 2 — constitutive local projection.**  
Probably notebook-only because Chapter 4 owns the detailed $\Delta\ell_r$ mechanics. Chapter 7 should avoid duplicating that chapter unless the synthesis requires a compact cross-harmonic example.

**Candidate 3 — geometry spectral projection.**  
Probably notebook-only because Chapter 5 already owns the detailed redistribution result.

**Candidate 4 — compliance zero-frequency projection.**  
Probably notebook-only because Chapter 6 owns the streaming calculation.

**Candidate 5 — VascuQuest branch-context figure.**  
Potentially strong Chapter 7 candidate. Its scientific value is precisely that it shows three kinds of physiological context side by side while refusing to collapse them into one model or score.

**Candidate 6 — age-group branch context.**  
Notebook-only unless execution reveals a particularly clear and nonredundant descriptive pattern.

The existing Chapter 7 nonlinear-unification schematic already carries the principal book-level synthesis. Any notebook-derived figure should supplement that schematic rather than duplicate it.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 7,
    "chapter_title": "Nonlinear Arterial Hemodynamics",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "constitutive_parameters": {
        "A_ztheta": A_ztheta,
        "A_thetaz": A_thetaz,
        "A_thetatheta": A_thetatheta,
    },
    "geometry_parameters": {
        "L_g": L_g,
        "alpha": alpha_G,
        "b_avg": b_G,
        "g_avg": g_G,
        "kappa_c": kappa_c,
    },
    "compliance_parameters": {
        "R_m": R_B,
        "f_Hz": f_B,
        "kR": kR_B,
        "P_Pa": P_B,
    },
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "sites": SITES,
    "constitutive_counterfactual_ratio_off": constitutive_ratio_off,
    "geometry_R_spec_full": float(Rspec_full),
    "geometry_R_spec_nonlinearity_off": float(Rspec_linear),
    "compliance_Q_stream_full_m3_s": float(Q_full),
    "compliance_Q_stream_sources_removed_m3_s": float(Q_off),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "The notebook compares three branch-specific equations and observables. "
        "It does not construct an additive anisotropy-geometry-compliance PDE, "
        "a unified patient-specific state, or a scalar nonlinearity index. "
        "VascuQuest supplies only branch-relevant physiological context."
    ),
}
(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- the algebraic harmonic-mixer demonstration;
- the canonical constitutive local-projection result;
- the canonical geometry spectral-redistribution result;
- the canonical compliant zero-frequency mean result;
- the branch counterfactual audit;
- VascuQuest population context for $\alpha$ and $\epsilon_{\mathrm{VQ}}$;
- representative source-geometry descriptors;
- age-group descriptive context;
- VascuQuest/PWDB verification metadata;
- a final reproducibility manifest.

The notebook deliberately contains no fourth mechanism, no Case D, no hidden branch coupling, and no unsupported cross-branch scalar diagnostic.